# Xselo no Colab: treinar numa GPU de verdade

Este notebook treina o **Xselo** em cima de um **Qwen2.5 de 7B, 14B ou 32B**, usando a GPU do Colab. Depois ele roda a prova fixa (com a parte de escrita), deixa você conversar com o modelo e salva o resultado.

- **0.4** (padrão): a **prosa é o ponto forte**. Treina com o lote de escrita caprichada (`dados_prosa/`), assuntos gerais com mais peso, e Touhou como um dos temas, não o único.
- **0.3**: a versão anterior (Touhou + assuntos gerais + matemática).

**Antes de começar:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → A100**. Se a A100 não estiver disponível, a L4 ou a T4 também servem, em 4 bits e com modelos menores.

Depois é só usar **Ambiente de execução → Executar tudo** (Ctrl + F9).

- **Pode rodar de novo sem medo:** se o modelo já estiver treinado nesta sessão ou salvo no seu Drive, o treino é pulado.
- Logo depois do treino e do polimento, uma cópia vai pro seu Drive (se ele estiver conectado).
- Na caixinha `você>`, converse à vontade; digite `/sair` pra ele seguir e baixar o `.zip`.

| modelo | GPU mínima | tempo estimado do treino da 0.4 na A100 | adaptador |
|---|---|---|---|
| Qwen2.5-7B | T4 (4 bits) | ~15–25 min | ~80 MB |
| Qwen2.5-14B | L4 ou A100 (4 bits) | ~25–45 min | ~140 MB |
| **Qwen2.5-32B** | A100 (4 bits) | **~45–90 min** | ~270 MB |

Os tempos são estimativa: a primeira rodada vai dizer o real. A prova no fim também leva um tempo (36 perguntas + 8 textos). Confira o saldo de unidades de computação do Colab antes do 32B.

In [ ]:
#@title ⚙️ 1. Configuração
VERSAO = "0.4"  #@param ["0.4", "0.3"]
MODELO = "Qwen/Qwen2.5-32B-Instruct"  #@param ["Qwen/Qwen2.5-32B-Instruct", "Qwen/Qwen2.5-14B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"]
QUATRO_BITS = "auto"  #@param ["auto", "sim", "não"]
EPOCAS = 1  #@param {type:"number"}
RANK = 16  #@param {type:"integer"}
SALVAR_NO_DRIVE = True  #@param {type:"boolean"}
POLIR_COM_DPO = True  #@param {type:"boolean"}

import re
TAMANHO_B = float(re.search(r"([\d.]+)B", MODELO).group(1))   # 7B -> 7.0
TAG = f"qwen{TAMANHO_B:g}b"                                       # qwen7b, qwen14b...
NOME = f"xselo-{VERSAO.replace('.', '-')}-{TAG}"                  # xselo-0-4-qwen32b
SAIDA = f"/content/saida/{NOME}"
print(f"Xselo {VERSAO} | modelo: {MODELO} ({TAMANHO_B:g}B) | saída: {SAIDA}")

In [ ]:
#@title 🖥️ 2. Conferir a GPU e decidir 4 bits / memória
import subprocess, torch
if not torch.cuda.is_available():
    raise SystemExit("Sem GPU! Vá em Ambiente de execução → Alterar o tipo de ambiente de execução → A100 (ou L4/T4).")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30

# memória aproximada só dos pesos: 2 bytes/parâmetro em bf16, ~0,6 em 4 bits (+ folga pro treino)
precisa_bf16 = TAMANHO_B * 2 * 1.4 + 6
precisa_4bit = TAMANHO_B * 0.6 + 8
if QUATRO_BITS == "auto":
    USAR_4BIT = VRAM < precisa_bf16
else:
    USAR_4BIT = QUATRO_BITS == "sim"
precisa = precisa_4bit if USAR_4BIT else precisa_bf16
CHECKPOINTING = TAMANHO_B >= 7        # gasta ~30% mais tempo, mas evita "falta de memória"
print(f"GPU com {VRAM:.0f} GB | 4 bits: {'sim' if USAR_4BIT else 'não'} | precisa de ~{precisa:.0f} GB "
      f"| gradient checkpointing: {'sim' if CHECKPOINTING else 'não'}")
if VRAM < precisa:
    raise SystemExit(f"{MODELO} não cabe nesta GPU ({VRAM:.0f} GB, precisa de ~{precisa:.0f} GB). "
                     "Escolha um modelo menor na célula 1 ou uma GPU maior (A100).")
print("✅ cabe!")

In [ ]:
#@title 📦 3. Baixar o código do Ratex e instalar as bibliotecas
import os
if os.path.isdir("/content/Ratex/.git"):
    !git -C /content/Ratex pull -q
else:
    !git clone -q --depth 1 https://github.com/SeniorVortex/Ratex.git /content/Ratex
!pip install -q -U transformers peft accelerate bitsandbytes safetensors huggingface_hub
!git -C /content/Ratex log --oneline -1

In [ ]:
#@title 💾 4. (Opcional) Conectar o Google Drive, pra não perder o modelo se a sessão cair
# Se aparecer uma janelinha pedindo permissão, escolha sua conta e clique em "Continuar"/"Permitir".
# Se o Drive falhar, tudo bem: o treino segue e no fim o modelo é baixado como .zip.
PASTA_DRIVE = None
if SALVAR_NO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PASTA_DRIVE = f"/content/drive/MyDrive/Ratex/{NOME}"
        print("✅ vai salvar em", PASTA_DRIVE)
    except Exception as erro:
        print(f"⚠️ não deu pra conectar o Drive ({erro}). Sem problema: seguindo sem ele.")
        print("   (pra tentar de novo depois, rode só esta célula e aceite a janelinha de permissão)")

In [ ]:
#@title 🏋️ 5. Treinar (a parte demorada; a loss tem que ir caindo)
import os, shutil
def ja_treinado(pasta):
    return pasta and os.path.exists(f"{pasta}/adapter_model.safetensors")

def guardar_no_drive(pasta, nome):
    if PASTA_DRIVE and ja_treinado(pasta):
        destino = os.path.join(os.path.dirname(PASTA_DRIVE), nome)
        shutil.copytree(pasta, destino, dirs_exist_ok=True)
        print("💾 cópia de segurança no Drive:", destino)

if ja_treinado(SAIDA):
    print("✅ o modelo já está treinado nesta sessão, pulando o treino:", SAIDA)
elif PASTA_DRIVE and ja_treinado(PASTA_DRIVE):
    shutil.copytree(PASTA_DRIVE, SAIDA, dirs_exist_ok=True)
    print("✅ achei o modelo no seu Drive, pulando o treino:", PASTA_DRIVE)
else:
    opcoes = f"--base {MODELO} --epocas {EPOCAS} --rank {RANK} --alpha {2 * RANK} --saida {SAIDA}"
    if USAR_4BIT:
        opcoes += " --4bit"
    if CHECKPOINTING:
        opcoes += " --checkpointing"
    print(f"python treinar_lora.py --versao {VERSAO}", opcoes, "\n")
    !cd /content/Ratex && python treinar_lora.py --versao {VERSAO} {opcoes}
    guardar_no_drive(SAIDA, NOME)

In [ ]:
#@title ✨ 5b. Polir com DPO (ensina a preferir as respostas boas de dados_preferencia/)
# Parar de inventar, aceitar correção quando você está certo, não concordar com informação errada.
# Leva poucos minutos. Daqui pra frente, as células usam o modelo polido.
if POLIR_COM_DPO:
    SAIDA_DPO, NOME_DPO = f"{SAIDA}-dpo", f"{NOME}-dpo"
    if ja_treinado(SAIDA_DPO):
        print("✅ já polido nesta sessão:", SAIDA_DPO)
    else:
        extra = "--checkpointing" if CHECKPOINTING else ""
        !cd /content/Ratex && python treinar_dpo.py --modelo {SAIDA} --saida {SAIDA_DPO} {extra}
        guardar_no_drive(SAIDA_DPO, NOME_DPO)
    if ja_treinado(SAIDA_DPO):
        SAIDA, NOME = SAIDA_DPO, NOME_DPO
        if PASTA_DRIVE:
            PASTA_DRIVE = os.path.join(os.path.dirname(PASTA_DRIVE), NOME)
        print("👉 usando o modelo polido:", SAIDA)
else:
    print("(polimento desligado na célula 1)")

In [ ]:
#@title 📝 6. Prova fixa (36 perguntas + 8 pedidos de escrita)
!cd /content/Ratex && python avaliar.py --modelo {SAIDA} --salvar {SAIDA}/prova.json
print("\nPra comparar, o Xselo 0.3 no Qwen 1.5B (o que está no GitHub) tirou 91,7 nas perguntas.")
print("A parte de prosa não vira nota: leia os textos! Estão em", f"{SAIDA}/prova.json", "(campo 'prosa').")

In [ ]:
#@title 💬 7. Conversar com o Xselo (digite /sair pra parar; pede uma crônica, um poema, um conselho...)
import os, sys
os.chdir("/content/Ratex"); sys.path.insert(0, "/content/Ratex")
from nucleo.hibrido import carregar_hibrido, responder
from nucleo.memoria import Memoria

if "xselo" not in globals():
    xselo, tok, cfg = carregar_hibrido(SAIDA, device="cuda")
    memoria = Memoria()
historico = []
while True:
    msg = input("você> ").strip()
    if msg in ("/sair", "sair", ""):
        break
    historico.append({"role": "user", "content": msg})
    print("xselo> ", end="")
    resposta = responder(xselo, tok, historico, system_prompt=cfg["system_prompt"], memoria=memoria, **cfg["geracao"])
    historico.append({"role": "assistant", "content": resposta})
    historico = historico[-12:]
    print()

In [ ]:
#@title 📤 8. Salvar: adaptador em float16 (metade do tamanho), Google Drive e .zip pra baixar
import shutil, os
from safetensors.torch import load_file, save_file
arq = f"{SAIDA}/adapter_model.safetensors"
pesos = load_file(arq)
save_file({k: v.half() for k, v in pesos.items()}, arq, metadata={"format": "pt"})
print(f"adaptador: {os.path.getsize(arq) / 2**20:.0f} MB")

if PASTA_DRIVE:
    shutil.copytree(SAIDA, PASTA_DRIVE, dirs_exist_ok=True)
    print("✅ copiado pro Drive:", PASTA_DRIVE)
else:
    print("(sem Drive conectado: guarde bem o .zip que vai baixar agora)")
zip_path = shutil.make_archive(f"/content/{NOME}", "zip", SAIDA)
print("zip pronto:", zip_path)
from google.colab import files
files.download(zip_path)

### 🤗 9. (Opcional) Publicar no Hugging Face

O jeito certo de guardar e compartilhar o modelo (e o único pros adaptadores de 14B/32B, que passam do limite de 100 MB do GitHub).

1. Crie uma conta em huggingface.co e um **token de escrita** (Settings → Access Tokens → New token, tipo *Write*).
2. No Colab, clique na **chave 🔑** na barra da esquerda (Secrets) e adicione `HF_TOKEN` com o token. Nunca cole o token direto no código.
3. Rode a célula abaixo.

Se deixar o repositório **público**, o Claude consegue baixar e integrar ele no GitHub do Ratex.

In [ ]:
#@title 🤗 9. Enviar pro Hugging Face
REPO_PUBLICO = False  #@param {type:"boolean"}
from google.colab import userdata
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
usuario = api.whoami()["name"]
repo = f"{usuario}/{NOME}"
api.create_repo(repo, private=not REPO_PUBLICO, exist_ok=True)
api.upload_folder(folder_path=SAIDA, repo_id=repo, commit_message=f"Xselo {VERSAO} em cima do {MODELO}")
print(f"✅ publicado: https://huggingface.co/{repo}")